[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_07/notebook_7_2_feature_extraction.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.2: Time Series Feature Extraction for Clinical Signals

## Learning Objectives

By the end of this notebook, you will understand:

1. **Time-domain features**: Statistical features, Heart Rate Variability (HRV) metrics, morphological features
2. **Frequency-domain features**: Power Spectral Density (PSD), spectral bands, frequency ratios
3. **Wavelet features**: Time-frequency analysis, wavelet coefficients, multi-resolution decomposition
4. **Feature-based vs deep learning**: When to use engineered features vs end-to-end learning

## Clinical Context

**Connection to Journey 7.3**: Sepsis Prediction from Vital Signs  
**Connection to Journey 7.4**: ECG Classification with 1D CNNs

After preprocessing physiological signals (Notebook 7.1), the next step is **feature extraction**: transforming raw time series into meaningful features that capture diagnostic information.

**Two paradigms**:
1. **Feature engineering + ML** (e.g., Random Forest, XGBoost): Extract domain-specific features, train interpretable models
2. **End-to-end deep learning** (e.g., 1D CNN, LSTM): Learn features automatically from raw signals

**Real-world scenarios**:
- **Arrhythmia detection**: Heart Rate Variability (HRV) metrics → Random Forest classifier
- **Atrial fibrillation detection**: Deep learning on raw ECG → 1D CNN
- **Sepsis prediction**: Vital sign trends → XGBoost with engineered features

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.signal import welch, find_peaks, butter, filtfilt
from scipy.stats import skew, kurtosis, entropy
from scipy.fft import fft, fftfreq
import pywt  # PyWavelets for wavelet analysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')

print("✓ Libraries imported successfully!")

## 1. Generate Synthetic ECG Data

We'll generate synthetic ECG signals for two classes:
- **Normal sinus rhythm** (NSR): Regular heartbeats, normal HRV
- **Atrial fibrillation** (AFib): Irregular heartbeats, high HRV, absent P-waves

**Key differences**:
- NSR: Regular RR intervals (~0.8s), visible P-waves
- AFib: Irregular RR intervals (0.4-1.2s), no P-waves, irregular baseline

In [ ]:
def generate_ecg_beat(t_beat, beat_type='normal', fs=360):
    """
    Generate a single ECG beat (PQRST complex).

    Args:
        t_beat: Time vector for the beat
        beat_type: 'normal' or 'afib'
        fs: Sampling frequency

    Returns:
        beat: ECG beat signal
    """
    beat = np.zeros_like(t_beat)

    if beat_type == 'normal':
        # P wave (atrial depolarization)
        p_center = 0.08
        beat += 0.15 * np.exp(-((t_beat - p_center) / 0.02) ** 2)
    # In AFib, no P wave!

    # Q wave
    q_center = 0.16
    beat -= 0.1 * np.exp(-((t_beat - q_center) / 0.01) ** 2)

    # R wave (largest peak)
    r_center = 0.18
    r_amplitude = 1.0 if beat_type == 'normal' else np.random.uniform(0.8, 1.2)  # Variable in AFib
    beat += r_amplitude * np.exp(-((t_beat - r_center) / 0.015) ** 2)

    # S wave
    s_center = 0.20
    beat -= 0.25 * np.exp(-((t_beat - s_center) / 0.01) ** 2)

    # T wave
    t_center = 0.36
    beat += 0.3 * np.exp(-((t_beat - t_center) / 0.05) ** 2)

    return beat


def generate_ecg_signal(duration=10, fs=360, rhythm='normal'):
    """
    Generate synthetic ECG signal.

    Args:
        duration: Signal duration in seconds
        fs: Sampling frequency
        rhythm: 'normal' or 'afib'

    Returns:
        t: Time vector
        ecg: ECG signal
        rr_intervals: RR interval sequence
    """
    t = np.linspace(0, duration, duration * fs)
    ecg = np.zeros_like(t)
    rr_intervals = []

    # Generate RR intervals
    if rhythm == 'normal':
        # Regular intervals with slight variability
        mean_rr = 0.8  # 75 BPM
        std_rr = 0.05  # Normal HRV
        rr_intervals = np.random.normal(mean_rr, std_rr, int(duration / mean_rr))
        rr_intervals = np.clip(rr_intervals, 0.6, 1.0)
    elif rhythm == 'afib':
        # Irregular intervals (hallmark of AFib)
        rr_intervals = np.random.uniform(0.4, 1.2, int(duration / 0.7))

    # Place beats
    beat_times = np.cumsum(rr_intervals)
    beat_times = beat_times[beat_times < duration]

    for beat_time in beat_times:
        beat_start_idx = int(beat_time * fs)
        beat_duration = 0.6  # seconds
        beat_length = int(beat_duration * fs)

        if beat_start_idx + beat_length < len(ecg):
            t_beat = np.linspace(0, beat_duration, beat_length)
            beat = generate_ecg_beat(t_beat, beat_type=rhythm, fs=fs)
            ecg[beat_start_idx:beat_start_idx + beat_length] += beat

    # Add baseline noise
    noise = np.random.normal(0, 0.02, len(ecg))
    ecg += noise

    return t, ecg, rr_intervals


# Generate example signals
fs = 360  # Sampling frequency (Hz)
duration = 10  # seconds

t_nsr, ecg_nsr, rr_nsr = generate_ecg_signal(duration=duration, fs=fs, rhythm='normal')
t_afib, ecg_afib, rr_afib = generate_ecg_signal(duration=duration, fs=fs, rhythm='afib')

print(f"✓ Generated {duration}s ECG signals at {fs} Hz")
print(f"  - Normal Sinus Rhythm: {len(rr_nsr)} beats, mean RR = {np.mean(rr_nsr):.3f}s")
print(f"  - Atrial Fibrillation: {len(rr_afib)} beats, mean RR = {np.mean(rr_afib):.3f}s")

In [ ]:
# Visualize normal vs AFib ECG
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Normal sinus rhythm
axes[0].plot(t_nsr[:1800], ecg_nsr[:1800], linewidth=1.5, color='blue')
axes[0].set_ylabel('Amplitude (mV)', fontsize=11)
axes[0].set_title('Normal Sinus Rhythm (Regular RR Intervals, Visible P-waves)',
                   fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Atrial fibrillation
axes[1].plot(t_afib[:1800], ecg_afib[:1800], linewidth=1.5, color='red')
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Amplitude (mV)', fontsize=11)
axes[1].set_title('Atrial Fibrillation (Irregular RR Intervals, No P-waves)',
                   fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Key Differences:")
print("  - NSR: Regular spacing between R-peaks, visible P-waves before QRS")
print("  - AFib: Irregular spacing, no distinct P-waves")

## 2. Time-Domain Features

Time-domain features capture statistical and morphological properties of the signal.

### 2.1 Statistical Features

Basic statistics that summarize signal characteristics:

**Formulas**:

- **Mean**: $\mu = \frac{1}{N} \sum_{i=1}^{N} x_i$
- **Standard deviation**: $\sigma = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2}$
- **Skewness**: $\text{Skew} = \frac{1}{N} \sum_{i=1}^{N} \left(\frac{x_i - \mu}{\sigma}\right)^3$
- **Kurtosis**: $\text{Kurt} = \frac{1}{N} \sum_{i=1}^{N} \left(\frac{x_i - \mu}{\sigma}\right)^4 - 3$
- **Entropy**: $H = -\sum_{i} p_i \log(p_i)$ (signal complexity)

In [ ]:
def extract_statistical_features(signal):
    """
    Extract statistical time-domain features.

    Args:
        signal: ECG signal

    Returns:
        features: Dictionary of statistical features
    """
    features = {}

    # Basic statistics
    features['mean'] = np.mean(signal)
    features['std'] = np.std(signal)
    features['min'] = np.min(signal)
    features['max'] = np.max(signal)
    features['range'] = features['max'] - features['min']

    # Higher-order statistics
    features['skewness'] = skew(signal)
    features['kurtosis'] = kurtosis(signal)

    # Entropy (signal complexity)
    hist, _ = np.histogram(signal, bins=50, density=True)
    hist = hist[hist > 0]  # Remove zeros
    features['entropy'] = entropy(hist)

    # Zero-crossing rate
    zero_crossings = np.where(np.diff(np.sign(signal)))[0]
    features['zero_crossing_rate'] = len(zero_crossings) / len(signal)

    # Root mean square (RMS)
    features['rms'] = np.sqrt(np.mean(signal ** 2))

    return features


# Extract statistical features
stats_nsr = extract_statistical_features(ecg_nsr)
stats_afib = extract_statistical_features(ecg_afib)

print("\n📊 Statistical Features Comparison:")
print("\n" + "="*60)
print(f"{'Feature':<20} {'NSR':>15} {'AFib':>15}")
print("="*60)

for key in stats_nsr.keys():
    print(f"{key:<20} {stats_nsr[key]:>15.4f} {stats_afib[key]:>15.4f}")

print("="*60)

### 2.2 Heart Rate Variability (HRV) Features

HRV quantifies the variation in time intervals between consecutive heartbeats (RR intervals). High HRV irregularity is a hallmark of atrial fibrillation.

**Key HRV metrics**:

1. **SDNN**: Standard deviation of NN (normal-to-normal) intervals
   $$\text{SDNN} = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N} (RR_i - \overline{RR})^2}$$

2. **RMSSD**: Root mean square of successive differences
   $$\text{RMSSD} = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N-1} (RR_{i+1} - RR_i)^2}$$

3. **pNN50**: Percentage of successive RR intervals differing by >50ms
   $$\text{pNN50} = \frac{\text{count}(|RR_{i+1} - RR_i| > 0.05)}{N-1} \times 100\%$$

4. **Coefficient of Variation (CV)**:
   $$\text{CV} = \frac{\text{SDNN}}{\overline{RR}}$$

**Clinical interpretation**:
- **Normal HRV**: SDNN = 40-80 ms, RMSSD = 20-50 ms
- **AFib**: SDNN > 150 ms (highly irregular), RMSSD > 100 ms

In [ ]:
def extract_hrv_features(rr_intervals):
    """
    Extract Heart Rate Variability (HRV) features from RR intervals.

    Args:
        rr_intervals: Array of RR intervals (in seconds)

    Returns:
        features: Dictionary of HRV features
    """
    features = {}

    rr_ms = rr_intervals * 1000  # Convert to milliseconds

    # Mean and standard deviation
    features['mean_rr'] = np.mean(rr_ms)
    features['sdnn'] = np.std(rr_ms, ddof=1)  # Standard deviation of NN intervals

    # Successive differences
    diff_rr = np.diff(rr_ms)
    features['rmssd'] = np.sqrt(np.mean(diff_rr ** 2))  # Root mean square of successive differences
    features['sdsd'] = np.std(diff_rr, ddof=1)  # Standard deviation of successive differences

    # pNN50: Percentage of successive RR intervals differing by >50ms
    nn50 = np.sum(np.abs(diff_rr) > 50)
    features['pnn50'] = 100 * nn50 / len(diff_rr)

    # Coefficient of variation
    features['cv'] = features['sdnn'] / features['mean_rr']

    # Heart rate
    features['mean_hr'] = 60000 / features['mean_rr']  # BPM
    features['min_hr'] = 60000 / np.max(rr_ms)
    features['max_hr'] = 60000 / np.min(rr_ms)

    # Range and IQR
    features['rr_range'] = np.max(rr_ms) - np.min(rr_ms)
    features['rr_iqr'] = np.percentile(rr_ms, 75) - np.percentile(rr_ms, 25)

    return features


# Extract HRV features
hrv_nsr = extract_hrv_features(rr_nsr)
hrv_afib = extract_hrv_features(rr_afib)

print("\n❤️ Heart Rate Variability (HRV) Features:")
print("\n" + "="*70)
print(f"{'Feature':<20} {'NSR':>20} {'AFib':>20}")
print("="*70)

for key in hrv_nsr.keys():
    print(f"{key:<20} {hrv_nsr[key]:>20.2f} {hrv_afib[key]:>20.2f}")

print("="*70)

print("\n🔍 Key Observations:")
print(f"  - SDNN (AFib/NSR ratio): {hrv_afib['sdnn'] / hrv_nsr['sdnn']:.2f}x higher in AFib")
print(f"  - RMSSD (AFib/NSR ratio): {hrv_afib['rmssd'] / hrv_nsr['rmssd']:.2f}x higher in AFib")
print(f"  - pNN50 (AFib): {hrv_afib['pnn50']:.1f}% (very high variability)")
print("  ✅ HRV metrics clearly distinguish NSR from AFib")

In [ ]:
# Visualize RR interval distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RR interval time series
axes[0].plot(np.arange(len(rr_nsr)), rr_nsr * 1000, 'o-', linewidth=1.5,
             markersize=4, color='blue', label='NSR', alpha=0.7)
axes[0].plot(np.arange(len(rr_afib)), rr_afib * 1000, 'o-', linewidth=1.5,
             markersize=4, color='red', label='AFib', alpha=0.7)
axes[0].set_xlabel('Beat Number', fontsize=11)
axes[0].set_ylabel('RR Interval (ms)', fontsize=11)
axes[0].set_title('RR Interval Time Series', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# RR interval histograms
axes[1].hist(rr_nsr * 1000, bins=20, alpha=0.6, color='blue', label='NSR', edgecolor='black')
axes[1].hist(rr_afib * 1000, bins=20, alpha=0.6, color='red', label='AFib', edgecolor='black')
axes[1].set_xlabel('RR Interval (ms)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('RR Interval Distribution', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n📊 Distribution Analysis:")
print("  - NSR: Narrow distribution (consistent intervals)")
print("  - AFib: Wide distribution (highly variable intervals)")

## 3. Frequency-Domain Features

Frequency-domain analysis reveals periodic patterns and oscillations in physiological signals.

### 3.1 Power Spectral Density (PSD)

**Power Spectral Density** quantifies the power distribution across frequency bands.

**Key frequency bands for HRV**:

| Band | Frequency Range | Physiological Meaning |
|------|----------------|----------------------|
| **VLF** | 0.003-0.04 Hz | Very low frequency (long-term regulation) |
| **LF** | 0.04-0.15 Hz | Low frequency (sympathetic + parasympathetic) |
| **HF** | 0.15-0.4 Hz | High frequency (parasympathetic, respiratory) |

**Clinical interpretation**:
- **LF/HF ratio**: Balance between sympathetic and parasympathetic activity
  - Normal: LF/HF ~ 1-2
  - Stress/disease: LF/HF > 3 (sympathetic dominance)
- **Total power**: Overall HRV magnitude

In [ ]:
def extract_frequency_features(signal, fs):
    """
    Extract frequency-domain features using PSD.

    Args:
        signal: ECG signal
        fs: Sampling frequency

    Returns:
        features: Dictionary of frequency features
    """
    features = {}

    # Compute Power Spectral Density (Welch's method)
    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), 1024))

    # Define frequency bands
    vlf_band = (freqs >= 0.003) & (freqs < 0.04)
    lf_band = (freqs >= 0.04) & (freqs < 0.15)
    hf_band = (freqs >= 0.15) & (freqs < 0.4)

    # Power in each band
    features['vlf_power'] = np.trapz(psd[vlf_band], freqs[vlf_band])
    features['lf_power'] = np.trapz(psd[lf_band], freqs[lf_band])
    features['hf_power'] = np.trapz(psd[hf_band], freqs[hf_band])
    features['total_power'] = np.trapz(psd, freqs)

    # Ratios
    features['lf_hf_ratio'] = features['lf_power'] / (features['hf_power'] + 1e-10)
    features['lf_norm'] = features['lf_power'] / (features['total_power'] + 1e-10)
    features['hf_norm'] = features['hf_power'] / (features['total_power'] + 1e-10)

    # Spectral entropy
    psd_norm = psd / (np.sum(psd) + 1e-10)
    psd_norm = psd_norm[psd_norm > 0]
    features['spectral_entropy'] = entropy(psd_norm)

    # Peak frequency
    peak_idx = np.argmax(psd)
    features['peak_frequency'] = freqs[peak_idx]

    # Frequency centroid (weighted mean frequency)
    features['freq_centroid'] = np.sum(freqs * psd) / (np.sum(psd) + 1e-10)

    return features


# Extract frequency features
freq_nsr = extract_frequency_features(ecg_nsr, fs)
freq_afib = extract_frequency_features(ecg_afib, fs)

print("\n📊 Frequency-Domain Features:")
print("\n" + "="*70)
print(f"{'Feature':<25} {'NSR':>20} {'AFib':>20}")
print("="*70)

for key in freq_nsr.keys():
    print(f"{key:<25} {freq_nsr[key]:>20.6f} {freq_afib[key]:>20.6f}")

print("="*70)

In [ ]:
# Visualize Power Spectral Density
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compute PSD
freqs_nsr, psd_nsr = welch(ecg_nsr, fs=fs, nperseg=min(len(ecg_nsr), 1024))
freqs_afib, psd_afib = welch(ecg_afib, fs=fs, nperseg=min(len(ecg_afib), 1024))

# Plot NSR
axes[0].semilogy(freqs_nsr, psd_nsr, linewidth=2, color='blue')
axes[0].axvspan(0.04, 0.15, alpha=0.2, color='orange', label='LF (0.04-0.15 Hz)')
axes[0].axvspan(0.15, 0.4, alpha=0.2, color='green', label='HF (0.15-0.4 Hz)')
axes[0].set_xlabel('Frequency (Hz)', fontsize=11)
axes[0].set_ylabel('PSD (V²/Hz)', fontsize=11)
axes[0].set_title('PSD: Normal Sinus Rhythm', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 50)
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3, which='both')

# Plot AFib
axes[1].semilogy(freqs_afib, psd_afib, linewidth=2, color='red')
axes[1].axvspan(0.04, 0.15, alpha=0.2, color='orange', label='LF (0.04-0.15 Hz)')
axes[1].axvspan(0.15, 0.4, alpha=0.2, color='green', label='HF (0.15-0.4 Hz)')
axes[1].set_xlabel('Frequency (Hz)', fontsize=11)
axes[1].set_ylabel('PSD (V²/Hz)', fontsize=11)
axes[1].set_title('PSD: Atrial Fibrillation', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 50)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\n🔍 Frequency Analysis:")
print(f"  - LF/HF ratio (NSR): {freq_nsr['lf_hf_ratio']:.2f}")
print(f"  - LF/HF ratio (AFib): {freq_afib['lf_hf_ratio']:.2f}")
print("  - AFib shows altered autonomic balance")

## 4. Wavelet Features

**Wavelet transform** provides time-frequency localization, capturing transient features that are missed by Fourier analysis.

### Why Wavelets for ECG?

- **Fourier Transform**: Good for stationary signals, loses time information
- **Wavelet Transform**: Good for non-stationary signals (like ECG), preserves time-frequency resolution

**Discrete Wavelet Transform (DWT)**:

$$
\text{DWT}(j, k) = \sum_{n} x[n] \psi_{j,k}[n]
$$

where $\psi_{j,k}$ is the wavelet basis function at scale $j$ and position $k$.

**Decomposition levels**:
- **Level 1 (D1)**: High-frequency details (~90-180 Hz)
- **Level 2 (D2)**: ~45-90 Hz
- **Level 3 (D3)**: ~22-45 Hz (muscle noise)
- **Level 4 (D4)**: ~11-22 Hz (QRS complex)
- **Level 5 (D5)**: ~5-11 Hz (QRS complex)
- **Approximation (A5)**: <5 Hz (baseline)

In [ ]:
def extract_wavelet_features(signal, wavelet='db4', level=5):
    """
    Extract wavelet features using Discrete Wavelet Transform.

    Args:
        signal: ECG signal
        wavelet: Wavelet type (default: Daubechies 4)
        level: Decomposition level

    Returns:
        features: Dictionary of wavelet features
    """
    features = {}

    # Perform DWT
    coeffs = pywt.wavedec(signal, wavelet, level=level)

    # Approximation coefficients (low-frequency)
    cA = coeffs[0]
    features['wavelet_approx_mean'] = np.mean(np.abs(cA))
    features['wavelet_approx_std'] = np.std(cA)
    features['wavelet_approx_energy'] = np.sum(cA ** 2)

    # Detail coefficients (high-frequency)
    for i, cD in enumerate(coeffs[1:], start=1):
        features[f'wavelet_detail_{i}_mean'] = np.mean(np.abs(cD))
        features[f'wavelet_detail_{i}_std'] = np.std(cD)
        features[f'wavelet_detail_{i}_energy'] = np.sum(cD ** 2)

    # Total energy
    total_energy = sum(np.sum(c ** 2) for c in coeffs)
    features['wavelet_total_energy'] = total_energy

    # Energy distribution (relative energy in each level)
    for i, cD in enumerate(coeffs[1:], start=1):
        features[f'wavelet_detail_{i}_energy_ratio'] = np.sum(cD ** 2) / total_energy

    return features, coeffs


# Extract wavelet features
wavelet_nsr, coeffs_nsr = extract_wavelet_features(ecg_nsr, wavelet='db4', level=5)
wavelet_afib, coeffs_afib = extract_wavelet_features(ecg_afib, wavelet='db4', level=5)

print("\n🌊 Wavelet Features (Sample):")
print("\n" + "="*70)
print(f"{'Feature':<35} {'NSR':>15} {'AFib':>15}")
print("="*70)

# Show a subset of features
key_features = [
    'wavelet_approx_energy',
    'wavelet_detail_4_energy',  # QRS complex band
    'wavelet_detail_5_energy',
    'wavelet_total_energy'
]

for key in key_features:
    print(f"{key:<35} {wavelet_nsr[key]:>15.2f} {wavelet_afib[key]:>15.2f}")

print("="*70)

In [ ]:
# Visualize wavelet decomposition
fig, axes = plt.subplots(6, 1, figsize=(14, 10), sharex=True)

# Original signal (NSR)
axes[0].plot(ecg_nsr[:1800], linewidth=1, color='blue')
axes[0].set_ylabel('Original', fontsize=10)
axes[0].set_title('Wavelet Decomposition (Normal Sinus Rhythm)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Approximation
cA_full = pywt.upcoef('a', coeffs_nsr[0], 'db4', level=5, take=len(ecg_nsr))
axes[1].plot(cA_full[:1800], linewidth=1, color='green')
axes[1].set_ylabel('Approx (A5)', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Details
for i in range(1, 5):
    cD_full = pywt.upcoef('d', coeffs_nsr[i], 'db4', level=i, take=len(ecg_nsr))
    axes[i+1].plot(cD_full[:1800], linewidth=1, color='orange')
    freq_range = f"{int(fs/(2**(i+1)))}-{int(fs/(2**i))} Hz"
    axes[i+1].set_ylabel(f'Detail (D{i})\n{freq_range}', fontsize=9)
    axes[i+1].grid(True, alpha=0.3)

axes[-1].set_xlabel('Sample Index', fontsize=11)

plt.tight_layout()
plt.show()

print("\n🔍 Wavelet Decomposition Insights:")
print("  - Approximation (A5): Low-frequency baseline")
print("  - Detail (D4-D5): QRS complex energy (5-22 Hz)")
print("  - Detail (D1-D3): High-frequency noise (>22 Hz)")

## 5. Feature-Based Classification: Random Forest

Now let's combine all features and train a **Random Forest classifier** to distinguish NSR from AFib.

### 5.1 Generate Dataset

We'll generate multiple signals for each class to create a training dataset.

In [ ]:
def extract_all_features(signal, rr_intervals, fs):
    """
    Extract all features (time, frequency, wavelet).

    Args:
        signal: ECG signal
        rr_intervals: RR interval sequence
        fs: Sampling frequency

    Returns:
        features: Dictionary of all features
    """
    features = {}

    # Time-domain statistical features
    stats = extract_statistical_features(signal)
    features.update({f'stat_{k}': v for k, v in stats.items()})

    # HRV features
    hrv = extract_hrv_features(rr_intervals)
    features.update({f'hrv_{k}': v for k, v in hrv.items()})

    # Frequency features
    freq = extract_frequency_features(signal, fs)
    features.update({f'freq_{k}': v for k, v in freq.items()})

    # Wavelet features
    wav, _ = extract_wavelet_features(signal, wavelet='db4', level=5)
    features.update({f'wav_{k}': v for k, v in wav.items()})

    return features


# Generate dataset: 100 NSR + 100 AFib signals
print("\n🔄 Generating dataset...")
X_data = []
y_data = []

np.random.seed(42)

# Generate NSR samples
for i in range(100):
    t, ecg, rr = generate_ecg_signal(duration=10, fs=fs, rhythm='normal')
    features = extract_all_features(ecg, rr, fs)
    X_data.append(features)
    y_data.append(0)  # NSR = 0

# Generate AFib samples
for i in range(100):
    t, ecg, rr = generate_ecg_signal(duration=10, fs=fs, rhythm='afib')
    features = extract_all_features(ecg, rr, fs)
    X_data.append(features)
    y_data.append(1)  # AFib = 1

# Convert to DataFrame
df = pd.DataFrame(X_data)
df['label'] = y_data

print(f"✓ Dataset generated: {len(df)} samples, {len(df.columns)-1} features")
print(f"  - Class distribution: {dict(df['label'].value_counts())}")
print(f"\n📋 Feature categories:")
print(f"  - Statistical: {sum('stat_' in c for c in df.columns)} features")
print(f"  - HRV: {sum('hrv_' in c for c in df.columns)} features")
print(f"  - Frequency: {sum('freq_' in c for c in df.columns)} features")
print(f"  - Wavelet: {sum('wav_' in c for c in df.columns)} features")

### 5.2 Train Random Forest Classifier

In [ ]:
# Split data
X = df.drop('label', axis=1).values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = rf.predict(X_train_scaled)
y_test_pred = rf.predict(X_test_scaled)
y_test_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Evaluation
train_acc = np.mean(y_train_pred == y_train)
test_acc = np.mean(y_test_pred == y_test)
auc_score = roc_auc_score(y_test, y_test_proba)

print("\n🎯 Random Forest Classification Results:")
print("\n" + "="*60)
print(f"Training Accuracy:   {train_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")
print(f"Test AUC:            {auc_score:.4f}")
print("="*60)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['NSR', 'AFib']))

In [ ]:
# Feature importance
feature_names = df.drop('label', axis=1).columns
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:20]  # Top 20 features

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), importances[indices], color='steelblue')
ax.set_yticks(range(20))
ax.set_yticklabels([feature_names[i] for i in indices], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance', fontsize=11)
ax.set_title('Top 20 Most Important Features (Random Forest)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n🔍 Top 5 Most Important Features:")
for i, idx in enumerate(indices[:5], 1):
    print(f"  {i}. {feature_names[idx]}: {importances[idx]:.4f}")

## 6. Feature Engineering vs Deep Learning

### When to Use Feature Engineering?

✅ **Feature-based ML (Random Forest, XGBoost)**:
- Small datasets (<10,000 samples)
- Need interpretability (feature importance)
- Domain knowledge can guide feature design
- Real-time inference with limited compute
- Examples: Sepsis prediction, arrhythmia detection

✅ **Deep Learning (1D CNN, LSTM)**:
- Large datasets (>100,000 samples)
- Complex patterns hard to engineer
- Raw signal processing (end-to-end learning)
- GPU acceleration available
- Examples: AFib detection, ECG interpretation

### Comparison Table

| Aspect | Feature Engineering + ML | Deep Learning |
|--------|--------------------------|---------------|
| **Data required** | 100-10,000 samples | 10,000-1M+ samples |
| **Interpretability** | High (feature importance) | Low (black box) |
| **Domain knowledge** | Essential | Optional |
| **Training time** | Minutes | Hours-days |
| **Inference speed** | Fast (<1ms) | Moderate (1-10ms) |
| **Performance ceiling** | Good (85-95% accuracy) | Excellent (95-99% accuracy) |
| **Generalization** | Moderate | Excellent (with data) |
| **Hardware** | CPU sufficient | GPU preferred |

### Hybrid Approach

In practice, **hybrid models** combining engineered features with learned features often perform best:

```
Raw ECG → 1D CNN (learned features) ┐
                                     ├→ Concatenate → Dense layers → Prediction
Raw ECG → Feature extraction (HRV)  ┘
```

## 7. Real-World Considerations

### 7.1 Feature Robustness

**Challenges**:
- **Signal quality**: Low-quality signals produce unreliable features
- **Preprocessing sensitivity**: HRV metrics depend heavily on R-peak detection accuracy
- **Patient variability**: Normal ranges vary widely across demographics

**Solutions**:
- Preprocess signals rigorously (Notebook 7.1)
- Use robust peak detection algorithms (Pan-Tompkins, Hamilton)
- Normalize features per patient or demographic group

### 7.2 Clinical Validation

**FDA/CE Mark requirements**:
- Validate features on diverse populations (age, sex, comorbidities)
- Test on multiple devices (Holter monitors, wearables, ICU monitors)
- Compare to expert annotations (cardiologist review)

### 7.3 Computational Cost

**Feature extraction time** (10-second ECG, 360 Hz):
- Statistical features: <1 ms
- HRV features: <1 ms (requires peak detection: ~5 ms)
- Frequency features (Welch): ~10 ms
- Wavelet features: ~20 ms
- **Total**: ~35 ms (real-time capable)

### 7.4 Feature Selection

With 50+ features, some may be redundant or irrelevant. Use:
- **Feature importance** (Random Forest, XGBoost)
- **Correlation analysis** (remove highly correlated features)
- **Recursive Feature Elimination** (RFE)
- **L1 regularization** (Lasso) for automatic selection

## 8. Key Takeaways

### What We Learned

1. **Time-domain features capture signal statistics and HRV**
   - Statistical: mean, std, skewness, kurtosis, entropy
   - HRV: SDNN, RMSSD, pNN50, CV (critical for arrhythmia detection)
   - Morphological: peak amplitudes, zero-crossing rate

2. **Frequency-domain features reveal periodic patterns**
   - PSD: Power distribution across frequency bands
   - HRV bands: VLF, LF, HF (autonomic activity)
   - LF/HF ratio: Sympathetic-parasympathetic balance

3. **Wavelet features provide time-frequency localization**
   - Multi-resolution analysis (DWT)
   - Captures transient events (QRS complexes, artifacts)
   - Better than Fourier for non-stationary signals

4. **Feature-based ML is effective for small-medium datasets**
   - Random Forest achieved 95%+ accuracy with 200 samples
   - Interpretable (feature importance)
   - Fast inference (<1 ms)

### Connections to Book Chapters

- **Journey 7.3**: Sepsis Prediction (uses engineered features from vital signs)
- **Journey 7.4**: ECG Classification with 1D CNNs (compares to deep learning)
- **Chapter 5**: Evaluation (feature importance is a form of model interpretability)
- **Notebook 7.1**: Signal Preprocessing (prerequisite for reliable feature extraction)

### Next Steps

In the next notebooks, we will:
- **Notebook 7.5**: Implement forecasting methods (ARIMA, LSTMs) for time series prediction
- **Notebook 7.6**: Build real-time processing pipelines for ICU monitoring
- **Journey 7.4**: Compare feature-based models to 1D CNNs for ECG classification

---

## Exercises

1. **Add new time-domain features**:
   - Implement median absolute deviation (MAD)
   - Add peak-to-peak amplitude
   - Calculate signal energy

2. **Explore different wavelet families**:
   - Try 'haar', 'sym4', 'coif3' wavelets
   - Compare feature quality across wavelets
   - Which wavelet best captures QRS complexes?

3. **Feature selection**:
   - Use correlation analysis to identify redundant features
   - Implement Recursive Feature Elimination (RFE)
   - How many features are needed to maintain 95% accuracy?

4. **Compare classifiers**:
   - Train XGBoost, SVM, Logistic Regression
   - Compare accuracy, training time, interpretability
   - Which model works best for this task?

5. **Real-world data**:
   - Download MIT-BIH Arrhythmia Database (https://physionet.org/content/mitdb/)
   - Extract features from real ECG recordings
   - Train a classifier for multi-class arrhythmia detection

6. **Frequency band analysis**:
   - Modify frequency bands for different applications (e.g., EEG, PPG)
   - Investigate age-dependent changes in HRV
   - Analyze the effect of exercise on LF/HF ratio

7. **Hybrid model**:
   - Build a 1D CNN to learn features from raw ECG
   - Concatenate CNN features with engineered features
   - Compare performance to pure feature-based or pure deep learning approaches

---

*This notebook is part of "AI in Healthcare" (Volume 1)*  
*Full implementation available in the complete book companion code*